# 23N Field Tests #05 · Ask about a government scheme, in Gujarati

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft05-gujarati-scheme-assistant/ft05-gujarati-scheme-assistant/ft05_gujarati_scheme_assistant.ipynb)

**Question.** Can a small open model, running for free, answer a citizen's question about a government scheme
accurately from the official documents, in Gujarati as well as in English?

**Method.** Retrieval-augmented generation (RAG). The notebook downloads 8 official documents on 7 schemes
(PM-KISAN, Sukanya Samriddhi, PMJJBY, PMSBY, Ayushman Bharat PM-JAY, Atal Pension Yojana and Gujarat's Vahli Dikri
Yojana), splits them into passages and indexes them with a multilingual embedding model. For each question the top
passages go to a ~3B-parameter open model, which must answer in the language of the question. The same 15 questions
are asked in English and in Gujarati, and each answer is checked against the amount or age stated in the source
document. A second pass hands the model the right passage directly ("oracle" context), which separates search
failures from answer failures.

**Data.** Official documents from the Press Information Bureau (PIB), PFRDA, the Department of Financial Services
and Gujarat's Women and Child Development Department, downloaded live from the publishers. They are Government of
India and Government of Gujarat publications; we don't redistribute them. If a download fails, the notebook falls
back to short attributed excerpts in this repo and says so.

**Models and licences.** Generator: `Qwen/Qwen2.5-3B-Instruct`, *Qwen Research* licence (**non-commercial**).
CPU fallback: `Qwen/Qwen2.5-1.5B-Instruct`, Apache 2.0. Retriever: `intfloat/multilingual-e5-small`, MIT.
Fonts: Bricolage Grotesque, IBM Plex and Noto Sans Gujarati, all SIL OFL 1.1.

**Runtime.** About 10–15 min on a free Colab **T4 GPU** (Runtime → Change runtime type → T4). No logins, no paid keys.

*Independent study on public data by 23N AI Labs. Not affiliated with or endorsed by any government department or
scheme mentioned here. This is not advice on scheme eligibility: always check the official portal.*

## 1 · Setup

In [ ]:
# CELL: system_deps
# libraqm gives Pillow complex-script shaping, which Gujarati needs (matras and conjuncts).
!apt-get -qq install -y libraqm0 libfribidi0 > /dev/null 2>&1 || echo "apt not available, continuing"
%pip install -q gradio pypdf beautifulsoup4

In [ ]:
# CELL: config
import os, sys, io, re, json, math, time, textwrap, unicodedata, subprocess, warnings, statistics
import urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="Glyph .* missing")

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet/GPU: it loads the excerpt snapshot
# instead of downloading, and swaps both models for tiny deterministic stand-ins. Never post its output.
SMOKE_TEST = os.environ.get("FT05_SMOKE_TEST", "0") == "1"

# ---- Assumptions and knobs (all named, all printed with the result) ----
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"                 # ~3.1B params, Qwen Research licence (non-commercial)
CPU_FALLBACK_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Apache 2.0, used automatically if there is no GPU
EMBED_ID = "intfloat/multilingual-e5-small"           # MIT, XLM-R based multilingual retriever
CHUNK_WORDS = 120          # passage length in words
CHUNK_OVERLAP = 30         # words shared by neighbouring passages
TOP_K = 4                  # passages handed to the model per question
MAX_NEW_TOKENS = 256       # Gujarati costs many tokens per word in this tokenizer, so allow room
MAX_EXTRA_NUMBERS = 3      # an answer listing more than this many unrelated numbers is marked wrong (no shotgun answers)
MIN_DOC_CHARS = 400        # a download shorter than this is treated as failed
MAX_SNAPSHOT_DOCS = 2      # if more documents than this fall back to excerpts, the run is not representative
EN_ACC_FLOOR = 0.40        # below this English accuracy the pipeline is probably broken (sanity check)
EN_RETRIEVAL_FLOOR = 0.60  # below this English retrieval hit rate the index is probably broken (sanity check)

# ---- Where things live (this branch until it is merged) ----
REPO_REF = "ft05-gujarati-scheme-assistant"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"
SNAPSHOT_LOCAL = "data/scheme_excerpts.json"
SNAPSHOT_URL = f"{RAW}/ft05-gujarati-scheme-assistant/data/scheme_excerpts.json"
FT_NUM = "05"

# ---- Brand kit ----
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| matplotlib", matplotlib.__version__, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: snapshot excerpts and stand-in models. These results are NOT real. Do NOT post.")

In [ ]:
# CELL: fonts
# Gujarati text. IBM Plex has no Gujarati glyphs, so Gujarati strings are drawn with Noto Sans Gujarati (SIL OFL,
# from the Google Fonts GitHub repo; it also covers Latin, digits and the rupee sign). matplotlib < 3.11 doesn't
# shape complex scripts, so those strings are rendered with Pillow + libraqm (HarfBuzz) and pasted in as images.
from PIL import Image, ImageDraw, ImageFont, features
os.makedirs("fonts", exist_ok=True)
GU_FONT = "fonts/NotoSansGujarati-VF.ttf"
GU_FONT_URL = ("https://raw.githubusercontent.com/google/fonts/main/ofl/notosansgujarati/"
               "NotoSansGujarati%5Bwdth%2Cwght%5D.ttf")
if not os.path.exists(GU_FONT):
    urllib.request.urlretrieve(GU_FONT_URL, GU_FONT)
HAS_RAQM = features.check("raqm")
if not HAS_RAQM:
    print("WARNING: Pillow has no libraqm, so Gujarati will be drawn WITHOUT shaping.\n"
          "Fix: run `!apt-get install -y libraqm0 libfribidi0`, then Runtime → Restart session, then Run all.")
_font_cache = {}
def _pil_font(px, bold=False):
    key = (int(px), bold)
    if key not in _font_cache:
        layout = ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC
        f = ImageFont.truetype(GU_FONT, int(px), layout_engine=layout)
        try:
            f.set_variation_by_axes([600 if bold else 400, 100])   # axes: wght, wdth
        except Exception:
            pass
        _font_cache[key] = f
    return _font_cache[key]

def has_gu(s):
    return any("\u0A80" <= c <= "\u0AFF" for c in str(s))

def text_image(text, px, color=B.INK, bold=False, max_width=None, line_gap=1.3, max_lines=None):
    """Render text (any script) to an RGBA array with proper shaping, wrapped to max_width pixels."""
    font = _pil_font(px, bold)
    lines = []
    for para in str(text).split("\n"):
        cur = ""
        for w in para.split(" "):
            trial = (cur + " " + w).strip()
            if max_width and cur and font.getlength(trial) > max_width:
                lines.append(cur); cur = w
            else:
                cur = trial
        lines.append(cur)
    if max_lines and len(lines) > max_lines:
        lines = lines[:max_lines]; lines[-1] = lines[-1].rstrip(" .,") + " …"
    lh = int(px * line_gap * 1.15)
    width = int(max(font.getlength(l) for l in lines)) + 8
    img = Image.new("RGBA", (max(width, 1), lh * len(lines) + int(px * 0.45)), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    for i, l in enumerate(lines):
        d.text((2, i * lh), l, font=font, fill=color)
    return np.asarray(img)

def put_text_img(fig, text, x, y, px, color=B.INK, bold=False, ha="left", max_width=None, alpha=1.0, max_lines=None):
    """Place shaped text on a figure. x, y = figure fraction of the top-left anchor. Returns height in figure fraction."""
    arr = text_image(text, px, color, bold, max_width, max_lines=max_lines)
    if alpha < 1:
        arr = arr.copy(); arr[..., 3] = (arr[..., 3] * alpha).astype(np.uint8)
    W, H = fig.get_size_inches() * fig.dpi
    h, w = arr.shape[:2]
    xo = x * W - (w / 2 if ha == "center" else (w if ha == "right" else 0))
    fig.figimage(arr, xo=int(xo), yo=int(y * H - h), origin="upper", zorder=10)
    return h / H

_chk = plt.figure(figsize=(9, 1.4), dpi=100)
put_text_img(_chk, "ગુજરાતી તપાસ: વહાલી દીકરી યોજનામાં કેટલી સહાય મળે છે? ₹૧,૧૦,૦૦૦", 0.02, 0.85, 32, B.INK, bold=True)
_chk.savefig("gujarati_render_check.png", facecolor=B.PAPER); plt.close(_chk)
print("libraqm shaping:", HAS_RAQM, "→ saved gujarati_render_check.png (open it to eyeball the conjuncts)")

## 2 · Data: 8 official documents, downloaded from the publishers

Each document is fetched from its official URL (PDF → text with `pypdf`, web page → text with BeautifulSoup). If a
download fails or returns almost nothing, the notebook uses the short excerpt for that document from
`data/scheme_excerpts.json` in this repo and flags it. The table below prints which source was used. Two
documents are deliberately about the same thing at different dates: the 2015 launch note gives the original
PMSBY/PMJJBY premiums, and the 2022 PIB release gives the revised ones. Citizens meet stale documents too.

In [ ]:
# CELL: documents
DOCS = [
 # id, scheme, short label, publisher, date, url
 ("D1", "PM-KISAN", "PM-KISAN backgrounder", "PIB", "2025-11",
  "https://static.pib.gov.in/WriteReadData/specificdocs/documents/2025/nov/doc20251120700401.pdf"),
 ("D2", "Sukanya Samriddhi", "Sukanya Samriddhi backgrounder", "PIB", "2026-01",
  "https://static.pib.gov.in/WriteReadData/specificdocs/documents/2026/jan/doc2026121762801.pdf"),
 ("D3", "PMJJBY", "PMJJBY: 10 years", "PIB", "2025-05",
  "https://static.pib.gov.in/WriteReadData/specificdocs/documents/2025/may/doc202558551501.pdf"),
 ("D4", "PM-JAY", "Six years of PM-JAY", "PIB / MoHFW", "2024-09",
  "https://static.pib.gov.in/WriteReadData/specificdocs/documents/2024/sep/doc2024923400001.pdf"),
 ("D5", "Atal Pension Yojana", "APY scheme page", "PFRDA", "live page",
  "https://pfrda.org.in/web/pfrda/schemes/atal-pension-yojana-apy"),
 ("D6", "PMSBY / PMJJBY / APY", "2015 launch press note", "Dept of Financial Services", "2015-05",
  "https://financialservices.gov.in/sites/default/files/press-release/Press-note-PMSBY-PMJJBY-APY.pdf"),
 ("D7", "PMSBY / PMJJBY", "2022 premium revision", "PIB (Ministry of Finance)", "2022-05",
  "https://pib.gov.in/PressReleasePage.aspx?PRID=1829772"),
 ("D8", "Vahli Dikri Yojana", "Vahli Dikri Yojana page", "Gujarat WCD", "live page",
  "https://wcd.gujarat.gov.in/posts?id=328"),
]
DOC_META = {d[0]: dict(zip(["id", "scheme", "label", "publisher", "date", "url"], d)) for d in DOCS}
UA = {"User-Agent": "Mozilla/5.0 (23N Field Tests research notebook; +https://23nlabs.com)"}

def _clean(t):
    t = unicodedata.normalize("NFC", t)
    t = re.sub(r"[ \t\u00a0]+", " ", t)
    t = re.sub(r"\n\s*\n+", "\n\n", t)
    return t.strip()

def fetch_text(url, timeout=40):
    import requests
    r = requests.get(url, headers=UA, timeout=timeout)
    r.raise_for_status()
    if url.lower().endswith(".pdf") or r.content[:4] == b"%PDF":
        from pypdf import PdfReader
        pdf = PdfReader(io.BytesIO(r.content))
        return _clean("\n".join((p.extract_text() or "") for p in pdf.pages))
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "noscript"]):
        tag.decompose()
    return _clean(soup.get_text("\n"))

def load_snapshot():
    if os.path.exists(SNAPSHOT_LOCAL):
        return json.load(open(SNAPSHOT_LOCAL, encoding="utf-8"))
    with urllib.request.urlopen(SNAPSHOT_URL, timeout=30) as f:
        return json.loads(f.read().decode("utf-8"))

def load_corpus_stand_in():
    """SMOKE_TEST only: no network. Uses the repo's excerpt snapshot for every document."""
    snap = load_snapshot()
    return {d: (snap["docs"][d]["text"], "snapshot (smoke test)") for d in DOC_META}

SNAP = None
CORPUS = {}
if SMOKE_TEST:
    CORPUS = load_corpus_stand_in()
else:
    for d, m in DOC_META.items():
        try:
            txt = fetch_text(m["url"])
            if len(txt) < MIN_DOC_CHARS:
                raise ValueError(f"only {len(txt)} chars")
            CORPUS[d] = (txt, "live")
        except Exception as e:
            SNAP = SNAP or load_snapshot()
            CORPUS[d] = (SNAP["docs"][d]["text"], f"snapshot excerpt (live failed: {type(e).__name__}: {str(e)[:60]})")

doc_table = pd.DataFrame([{**{k: DOC_META[d][k] for k in ("id", "scheme", "publisher", "date")},
                           "chars": len(CORPUS[d][0]), "source": CORPUS[d][1]} for d in DOC_META])
N_SNAPSHOT = int((~doc_table.source.eq("live")).sum())
N_DOCS = len(doc_table); N_SCHEMES = 7
print(doc_table.to_string(index=False))
print(f"\n{N_DOCS - N_SNAPSHOT}/{N_DOCS} documents downloaded live.")

In [ ]:
# CELL: chunking
def chunk_words(text, n=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    if len(words) <= n:
        return [" ".join(words)]
    out, step = [], n - overlap
    for i in range(0, len(words), step):
        out.append(" ".join(words[i:i + n]))
        if i + n >= len(words):
            break
    return out

PASSAGES = []   # dicts: pid, doc, text
for d, (txt, _) in CORPUS.items():
    for j, ch in enumerate(chunk_words(txt)):
        PASSAGES.append({"pid": f"{d}.{j}", "doc": d, "text": f"[{DOC_META[d]['label']}, {DOC_META[d]['publisher']}, {DOC_META[d]['date']}] {ch}"})
chunks_per_doc = pd.Series([p["doc"] for p in PASSAGES]).value_counts().reindex(list(DOC_META)).fillna(0).astype(int)
print(f"{len(PASSAGES)} passages from {N_DOCS} documents:", dict(chunks_per_doc))

fig, ax = plt.subplots(figsize=(9, 4.2))
labels = [f"{d} · {DOC_META[d]['label']}" for d in chunks_per_doc.index]
ax.barh(labels[::-1], chunks_per_doc.values[::-1], color=B.MOSS)
ax.set_xlabel("passages indexed")
B.chart_title(ax, "The retrieval corpus", f"{N_DOCS} official documents · {len(PASSAGES)} passages of ~{CHUNK_WORDS} words")
B.source_note(fig, "Sources: PIB, PFRDA, Dept of Financial Services, Gujarat WCD. Independent study on public data.")
plt.tight_layout(); plt.show()

## 3 · Models: a multilingual retriever and a ~3B open generator

`intfloat/multilingual-e5-small` turns passages (prefix `passage: `) and questions (prefix `query: `) into vectors,
so a Gujarati question can find an English passage. `Qwen/Qwen2.5-3B-Instruct` in fp16 needs about 6.5 GB of VRAM
and fits a free T4. Decoding is greedy, so the score is reproducible on the same hardware and library versions.

In [ ]:
# CELL: model_load
def load_models():
    global MODEL_ID
    import torch
    from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
    has_gpu = torch.cuda.is_available()
    dev = "cuda" if has_gpu else "cpu"
    if not has_gpu:
        print("No GPU found → switching to", CPU_FALLBACK_MODEL_ID, "(slower; enable a T4 for the headline run)")
        MODEL_ID = CPU_FALLBACK_MODEL_ID
    etok = AutoTokenizer.from_pretrained(EMBED_ID)
    emod = AutoModel.from_pretrained(EMBED_ID).to(dev).eval()
    def embed(texts, prefix, bs=32):
        vecs = []
        for i in range(0, len(texts), bs):
            batch = etok([prefix + t for t in texts[i:i + bs]], max_length=512, padding=True, truncation=True,
                         return_tensors="pt").to(dev)
            with torch.no_grad():
                h = emod(**batch).last_hidden_state
            m = batch["attention_mask"].unsqueeze(-1).float()
            v = (h * m).sum(1) / m.sum(1)
            vecs.append(torch.nn.functional.normalize(v, dim=-1).cpu().numpy())
        return np.vstack(vecs)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if has_gpu else torch.float32,
                                                 device_map="auto" if has_gpu else None).eval()
    def generate(messages, max_new_tokens=MAX_NEW_TOKENS):
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok([text], return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                                 top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    params = sum(p.numel() for p in model.parameters()) / 1e9
    hw = torch.cuda.get_device_name(0) if has_gpu else "CPU"
    return embed, generate, params, hw

def load_stand_in_models():
    """SMOKE_TEST only: character n-gram TF-IDF retrieval and an extractive stub that returns the first context
    sentence containing a number. Exists so the notebook can be executed end to end without internet or a GPU."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit([p["text"] for p in PASSAGES])
    def embed(texts, prefix):
        m = vec.transform(texts).toarray()
        return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)
    def generate(messages, max_new_tokens=MAX_NEW_TOKENS):
        body = messages[-1]["content"]
        ctx, q = body.split("Question:")[0], body.split("Question:")[-1]
        ctx = re.sub(r"\[S\d+\] \[[^\]]*\]", " ", ctx.replace("Passages:", ""))
        qw = set(re.findall(r"[a-z]{4,}", q.lower()))
        sents = [x.strip() for x in re.split(r"(?<=[.!?])\s+", ctx) if re.search(r"\d", x)]
        if not sents:
            return "I don't know."
        best = max(sents, key=lambda x: len(qw & set(re.findall(r"[a-z]{4,}", x.lower()))))
        return best[:220] + " [S1]"
    return embed, generate, 0.0, "CPU (stand-in)"

t0 = time.time()
embed, generate, PARAMS_B, HARDWARE = load_stand_in_models() if SMOKE_TEST else load_models()
MODEL_LABEL = "stand-in (smoke test)" if SMOKE_TEST else MODEL_ID.split("/")[-1]
_m = re.search(r"(\d+(?:\.\d+)?)B", MODEL_LABEL)
MODEL_SIZE = f"{_m.group(1)}B" if _m else "small"
EMBED_LABEL = EMBED_ID.split("/")[-1]
P_VECS = embed([p["text"] for p in PASSAGES], "passage: ")
print(f"Models ready in {time.time() - t0:.0f}s on {HARDWARE}: {MODEL_LABEL} ({PARAMS_B:.2f}B params) + {EMBED_LABEL}; "
      f"{P_VECS.shape[0]} passages embedded")

## 4 · Method: retrieve, then answer in the question's language

1. Embed the question and take the `TOP_K` most similar passages.
2. The prompt asks the model to answer only from those passages, in the language of the question (Gujarati script
   for a Gujarati question), to state the exact amount or age, to cite the passage, and to say it doesn't know if
   the passages don't say.
3. **Oracle variant.** The same question with the top passages from the correct document only. If the model fails
   here too, the failure is in reading or writing the answer, not in search.

In [ ]:
# CELL: rag
SYSTEM_PROMPT = (
    "You help citizens understand Indian government schemes. Answer ONLY from the numbered passages provided. "
    "Answer in the same language as the question: if the question is in Gujarati, answer in Gujarati script. "
    "State the exact amount, age or number from the passage, with its unit. If passages disagree, prefer the most "
    "recent one. Keep the answer to one or two sentences and cite the passage like [S1]. If the passages do not "
    "contain the answer, say you don't know.")

def retrieve(question, k=TOP_K, only_doc=None):
    q = embed([question], "query: ")[0]
    sims = P_VECS @ q
    order = np.argsort(-sims)
    if only_doc:
        order = [i for i in order if PASSAGES[i]["doc"] == only_doc]
    return [(PASSAGES[i], float(sims[i])) for i in order[:k]]

def ask(question, k=TOP_K, only_doc=None):
    hits = retrieve(question, k, only_doc)
    ctx = "\n\n".join(f"[S{i + 1}] {p['text']}" for i, (p, _) in enumerate(hits))
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {question}"}]
    t = time.time()
    ans = generate(msgs)
    return {"question": question, "answer": ans, "latency_s": time.time() - t,
            "docs": [p["doc"] for p, _ in hits], "pids": [p["pid"] for p, _ in hits]}

## 5 · Evaluation: the same 15 questions in English and Gujarati

Each question has a gold answer taken from one source document, written as the number(s) a correct answer must
contain. The scorer:

- converts Gujarati digits (૦–૯) to 0–9, drops digit-group commas (1,50,000 → 150000), and expands *lakh/લાખ*,
  *crore/કરોડ* and *thousand/હજાર* (₹2 lakh → 200000);
- marks an answer **correct** if every gold number appears, no "stale" number appears (the 2015 premiums), and it
  doesn't list more than `MAX_EXTRA_NUMBERS` other numbers;
- records separately whether the answer to a Gujarati question is written mainly in Gujarati script.

The Gujarati questions are 23N AI Labs' own translations of the English ones. Please report awkward wording.

In [ ]:
# CELL: eval_set
# id, gold doc, gold numbers, stale numbers that make an answer wrong, reference answer, English, Gujarati
EVAL = [
 ("Q01", "D1", [6000], [], "Rs 6,000 a year (three instalments of Rs 2,000)",
  "How much money does an eligible farmer family get in a year under PM-KISAN?",
  "પીએમ-કિસાન યોજના હેઠળ પાત્ર ખેડૂત પરિવારને વર્ષે કેટલા રૂપિયા મળે છે?"),
 ("Q02", "D2", [10], [], "Until the girl turns 10",
  "Up to what age of the girl can a Sukanya Samriddhi account be opened?",
  "દીકરીની કેટલી ઉંમર સુધી સુકન્યા સમૃદ્ધિ ખાતું ખોલાવી શકાય?"),
 ("Q03", "D2", [250], [], "Rs 250 each financial year",
  "What is the minimum amount that must be deposited each financial year in a Sukanya Samriddhi account?",
  "સુકન્યા સમૃદ્ધિ ખાતામાં દર નાણાકીય વર્ષે ઓછામાં ઓછી કેટલી રકમ જમા કરાવવી પડે?"),
 ("Q04", "D2", [150000], [], "Rs 1,50,000 a year",
  "What is the maximum amount that can be deposited in a Sukanya Samriddhi account in one year?",
  "સુકન્યા સમૃદ્ધિ ખાતામાં એક વર્ષમાં વધુમાં વધુ કેટલી રકમ જમા કરાવી શકાય?"),
 ("Q05", "D2", [21], [], "21 years from opening",
  "After how many years from opening does a Sukanya Samriddhi account mature?",
  "સુકન્યા સમૃદ્ધિ ખાતું ખોલાવ્યાના કેટલા વર્ષ પછી પાકે છે?"),
 ("Q06", "D3", [200000], [], "Rs 2 lakh life cover",
  "How much life insurance cover does PMJJBY give?",
  "પ્રધાનમંત્રી જીવન જ્યોતિ વીમા યોજના (PMJJBY) માં કેટલું જીવન વીમા કવચ મળે છે?"),
 ("Q07", "D3", [436], [330], "Rs 436 a year (revised from Rs 330 in 2022)",
  "What is the yearly premium for PMJJBY?",
  "PMJJBY નું વાર્ષિક પ્રીમિયમ કેટલું છે?"),
 ("Q08", "D3", [50], [], "50 years (cover continues to 55)",
  "What is the maximum age to enrol in PMJJBY?",
  "PMJJBY માં જોડાવા માટે વધુમાં વધુ ઉંમર કેટલી છે?"),
 ("Q09", "D7", [20], [12], "Rs 20 a year since 1 June 2022 (was Rs 12)",
  "What is the current yearly premium for PMSBY accident insurance?",
  "પ્રધાનમંત્રી સુરક્ષા વીમા યોજના (PMSBY) નું હાલનું વાર્ષિક પ્રીમિયમ કેટલું છે?"),
 ("Q10", "D4", [500000], [], "Rs 5 lakh per family per year",
  "How much health cover does a family get per year under Ayushman Bharat PM-JAY?",
  "આયુષ્માન ભારત PM-JAY હેઠળ એક પરિવારને વર્ષે કેટલું આરોગ્ય કવચ મળે છે?"),
 ("Q11", "D4", [70], [], "70 years and above",
  "From what age are senior citizens covered under PM-JAY regardless of income?",
  "આવકને ધ્યાનમાં લીધા વિના કેટલા વર્ષ કે તેથી વધુ ઉંમરના વરિષ્ઠ નાગરિકોને PM-JAY નો લાભ મળે છે?"),
 ("Q12", "D5", [18, 40], [], "Between 18 and 40",
  "Between what ages can someone join Atal Pension Yojana?",
  "અટલ પેન્શન યોજનામાં કઈ ઉંમરથી કઈ ઉંમર સુધી જોડાઈ શકાય?"),
 ("Q13", "D5", [5000], [], "Rs 5,000 a month",
  "What is the highest monthly pension under Atal Pension Yojana?",
  "અટલ પેન્શન યોજનામાં વધુમાં વધુ માસિક પેન્શન કેટલું મળે છે?"),
 ("Q14", "D8", [100000], [], "Rs 1,00,000 at age 18",
  "Under Vahli Dikri Yojana, how much is given when the daughter turns 18?",
  "વહાલી દીકરી યોજનામાં દીકરી 18 વર્ષની થાય ત્યારે કેટલી સહાય મળે છે?"),
 ("Q15", "D8", [200000], [], "Rs 2,00,000 or less a year",
  "What is the maximum annual income of the couple to be eligible for Vahli Dikri Yojana?",
  "વહાલી દીકરી યોજનાનો લાભ લેવા માટે દંપતીની વાર્ષિક આવક વધુમાં વધુ કેટલી હોવી જોઈએ?"),
]
N_Q = len(EVAL)
assert all(has_gu(q[6]) for q in EVAL) and not any(has_gu(q[5]) for q in EVAL)
print(N_Q, "questions × 2 languages")

In [ ]:
# CELL: scoring
GU_DIGITS = str.maketrans("૦૧૨૩૪૫૬૭૮૯", "0123456789")
MULT = {"lakh": 1e5, "lakhs": 1e5, "lac": 1e5, "lacs": 1e5, "લાખ": 1e5, "crore": 1e7, "crores": 1e7, "કરોડ": 1e7,
        "thousand": 1e3, "હજાર": 1e3}
_UNITS = "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split()
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90}
_WORD_RE = re.compile(r"\b(?:(" + "|".join(_TENS) + r")(?:[- ](" + "|".join(_UNITS[1:10]) + r"))?|(" + "|".join(_UNITS) + r"))\b", re.I)
_GU_WORDS = {"એક": 1, "બે": 2, "ત્રણ": 3, "ચાર": 4, "પાંચ": 5, "છ": 6, "સાત": 7, "આઠ": 8, "નવ": 9, "દસ": 10}
_GU_WORD_RE = re.compile(r"(?<![\u0A80-\u0AFF])(" + "|".join(sorted(_GU_WORDS, key=len, reverse=True)) + r")(?![\u0A80-\u0AFF])")

def _words_to_digits(s):
    def en(m):
        if m.group(3): return str(_UNITS.index(m.group(3).lower()))
        return str(_TENS[m.group(1).lower()] + (_UNITS.index(m.group(2).lower()) if m.group(2) else 0))
    return _GU_WORD_RE.sub(lambda m: str(_GU_WORDS[m.group(1)]), _WORD_RE.sub(en, s))

_MULT_RE = re.compile(r"(\d+(?:\.\d+)?)\s*(" + "|".join(sorted(MULT, key=len, reverse=True)) + r")", re.I)

def numbers_in(text):
    s = unicodedata.normalize("NFC", str(text)).translate(GU_DIGITS)
    s = _words_to_digits(re.sub(r"\[[^\]]*\]", " ", s))                 # drop citations like [S1] or [PIB, 2025-05]
    s = re.sub(r"(?<=\d)[,\u066c](?=\d)", "", s)       # 1,50,000 → 150000
    nums = {round(float(x), 2) for x in re.findall(r"\d+(?:\.\d+)?", s)}
    for x, unit in _MULT_RE.findall(s):
        nums.add(round(float(x) * MULT[unit.lower()], 2))
    return nums

def gu_share(text):
    letters = [c for c in str(text) if c.isalpha()]
    return sum(has_gu(c) for c in letters) / max(len(letters), 1)

def score(answer, question, gold, stale):
    got = numbers_in(answer)
    extra = got - set(map(float, gold)) - numbers_in(question)
    ok = all(float(g) in got for g in gold) and not any(float(s) in got for s in stale) and len(extra) <= MAX_EXTRA_NUMBERS
    return ok, sorted(got)

# self-test of the scorer on hand-made strings (not model output)
assert score("તમને વર્ષે ₹૬,૦૦૦ મળે છે [S1]", "", [6000], [])[0]
assert score("Rs. 2 lakh life cover", "", [200000], [])[0]
assert score("₹1.5 lakh", "", [150000], [])[0]
assert not score("The premium is Rs 12 (now Rs 20)", "", [20], [12])[0]
assert score("18 થી 40 વર્ષ", "", [18, 40], [])[0]
assert score("matures after twenty-one years", "", [21], [])[0]
assert score("બે લાખ રૂપિયા", "", [200000], [])[0]
assert score("Rupees Two Lakh", "", [200000], [])[0]
assert not score("1 2 3 4 5 6000", "", [6000], [])[0]
print("scorer self-test passed")

In [ ]:
# CELL: run_eval
ROWS = []
for qid, gdoc, gold, stale, ref, q_en, q_gu in EVAL:
    for lang, q in (("en", q_en), ("gu", q_gu)):
        for mode in ("rag", "oracle"):
            r = ask(q, only_doc=gdoc if mode == "oracle" else None)
            ok, nums = score(r["answer"], q, gold, stale)
            ROWS.append({"qid": qid, "lang": lang, "mode": mode, "question": q, "answer": r["answer"],
                         "correct": ok, "numbers": nums, "gold": gold, "reference": ref, "gold_doc": gdoc,
                         "retrieved_gold_doc": gdoc in r["docs"], "docs": r["docs"],
                         "gu_script_share": gu_share(r["answer"]), "latency_s": r["latency_s"]})
        rr = [x for x in ROWS if x["qid"] == qid and x["lang"] == lang and x["mode"] == "rag"][0]
        print(f"{qid} [{lang}] {'✓' if rr['correct'] else '✗'} {'R' if rr['retrieved_gold_doc'] else '-'} "
              f"{rr['latency_s']:.1f}s  {rr['answer'][:110]!r}")
res = pd.DataFrame(ROWS)
res.to_csv("ft05_eval_results.csv", index=False)
print("saved ft05_eval_results.csv;", len(res), "rows (15 questions × 2 languages × 2 modes)")

## 6 · Results: the headline number, after the sanity checks

In [ ]:
# CELL: results
rag = res[res["mode"] == "rag"]; ora = res[res["mode"] == "oracle"]
def _n(df, lang, col="correct"): return int(df[df["lang"] == lang][col].sum())
EN_CORRECT, GU_CORRECT = _n(rag, "en"), _n(rag, "gu")
EN_ORACLE, GU_ORACLE = _n(ora, "en"), _n(ora, "gu")
EN_RET, GU_RET = _n(rag, "en", "retrieved_gold_doc"), _n(rag, "gu", "retrieved_gold_doc")
GU_SCRIPT = int((rag[rag["lang"] == "gu"]["gu_script_share"] >= 0.5).sum())
LAT_MEDIAN = float(rag["latency_s"].median())
LAT_EN = float(rag[rag["lang"] == "en"]["latency_s"].median()); LAT_GU = float(rag[rag["lang"] == "gu"]["latency_s"].median())

# ---------------- sanity checks (before any headline is printed) ----------------
PROBLEMS = []
for qid, gdoc, gold, stale, ref, q_en, q_gu in EVAL:          # is every gold answer actually in its source text?
    have = numbers_in(CORPUS[gdoc][0])
    missing = [g for g in gold if float(g) not in have]
    if missing:
        PROBLEMS.append(f"{qid}: gold value(s) {missing} not found in {gdoc} as loaded; the document may have changed")
for name, v in [("EN", EN_CORRECT), ("GU", GU_CORRECT), ("EN oracle", EN_ORACLE), ("GU oracle", GU_ORACLE)]:
    if not 0 <= v <= N_Q:
        PROBLEMS.append(f"{name} correct = {v}, outside 0..{N_Q}")
if N_SNAPSHOT > MAX_SNAPSHOT_DOCS:
    PROBLEMS.append(f"{N_SNAPSHOT} of {N_DOCS} documents fell back to excerpts; retrieval over excerpts is easier than over full documents")
if EN_CORRECT / N_Q < EN_ACC_FLOOR:
    PROBLEMS.append(f"English accuracy {EN_CORRECT}/{N_Q} is below {EN_ACC_FLOOR:.0%}: check prompts, model load and scorer before blaming the language")
if EN_RET / N_Q < EN_RETRIEVAL_FLOOR:
    PROBLEMS.append(f"English retrieval found the right document for only {EN_RET}/{N_Q} questions: the index is probably broken")
if EN_ORACLE < EN_CORRECT - 2 or GU_ORACLE < GU_CORRECT - 2:
    PROBLEMS.append("Oracle context scored clearly worse than retrieval; that is implausible and suggests a bug in the oracle path")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE_TEST is on: stand-in data and models")
if N_SNAPSHOT:
    print(f"Note: {N_SNAPSHOT} document(s) came from the excerpt snapshot:", list(doc_table[doc_table.source != 'live'].id))
if PROBLEMS:
    print("!" * 100)
    for p in PROBLEMS: print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every check above passes.")
    print("!" * 100)
else:
    print("All sanity checks passed.")

def make_headline():
    link = "but only" if GU_CORRECT < EN_CORRECT else "and"
    return (f"A {MODEL_SIZE} open model answered {EN_CORRECT}/{N_Q} government-scheme questions correctly in English "
            f"{link} {GU_CORRECT}/{N_Q} in Gujarati, reading the same official documents on a free GPU.")
HEADLINE = make_headline()
print()
print("HEADLINE (estimate from a 15-question probe):", HEADLINE)
print(f"  Retrieval found the right document: EN {EN_RET}/{N_Q}, GU {GU_RET}/{N_Q}")
print(f"  Robustness, oracle context (right document handed over): EN {EN_ORACLE}/{N_Q}, GU {GU_ORACLE}/{N_Q}")
print(f"  Gujarati answers written mainly in Gujarati script: {GU_SCRIPT}/{N_Q}")
print(f"  Median latency: EN {LAT_EN:.1f}s, GU {LAT_GU:.1f}s on {HARDWARE}")
print(f"  Assumptions: TOP_K={TOP_K}, CHUNK_WORDS={CHUNK_WORDS}, CHUNK_OVERLAP={CHUNK_OVERLAP}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}, "
      f"MAX_EXTRA_NUMBERS={MAX_EXTRA_NUMBERS}; generator {MODEL_LABEL}; retriever {EMBED_LABEL}")

In [ ]:
# CELL: results_chart
def draw_results(ax, big=False):
    groups = ["Right document\nretrieved", "Correct answer\n(retrieval)", "Correct answer\n(oracle context)"]
    en = [EN_RET, EN_CORRECT, EN_ORACLE]; gu = [GU_RET, GU_CORRECT, GU_ORACLE]
    x = np.arange(len(groups)); w = 0.36
    b1 = ax.bar(x - w / 2, en, w, color=B.MOSS, label="English")
    b2 = ax.bar(x + w / 2, gu, w, color=B.AMBER, label="Gujarati")
    for bars in (b1, b2):
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.25, f"{int(b.get_height())}", ha="center",
                    va="bottom", fontsize=18 if big else 11, color=B.INK, fontproperties=B.FP.get("sans_bold"))
    ax.set_xticks(x, groups, fontsize=15 if big else 10)
    ax.set_ylim(0, N_Q + 2); ax.set_ylabel(f"out of {N_Q} questions", fontsize=14 if big else 10)
    ax.tick_params(axis="y", labelsize=13 if big else 10)
    ax.legend(loc="upper right", fontsize=15 if big else 10, ncol=2)

fig, ax = plt.subplots(figsize=(9, 4.6))
draw_results(ax)
B.chart_title(ax, "Same questions, two languages", f"{MODEL_LABEL} + {EMBED_LABEL} · {N_Q} questions each")
B.source_note(fig, "23N Field Test #05 · official scheme documents (PIB, PFRDA, DFS, Gujarat WCD) · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft05_results.png", dpi=150); plt.show()

show = rag.pivot(index="qid", columns="lang", values="correct").rename(columns={"en": "EN ✓", "gu": "GU ✓"})
show["reference"] = [e[4] for e in EVAL]
print(show.to_string())

## 7 · Interactive demo
Ask your own question in Gujarati or English. You get the answer and the passages it used. The cell doesn't block
"Run all"; open the public `gradio.live` link it prints. Answers can be wrong: check the official portal.

In [ ]:
# CELL: gradio
def gr_answer(question):
    question = (question or "").strip()
    if not question:
        return "Type a question first.", ""
    r = ask(question)
    src = "\n".join(f"- **{d}** {DOC_META[d]['label']} ({DOC_META[d]['publisher']}, {DOC_META[d]['date']}): {DOC_META[d]['url']}"
                    for d in dict.fromkeys(r["docs"]))
    return f"**{r['answer']}**\n\n<sub>{r['latency_s']:.1f}s · {MODEL_LABEL} · not advice: check the official portal</sub>", src

try:
    if SMOKE_TEST:
        raise RuntimeError("skipped in smoke test")
    import gradio as gr
    with gr.Blocks(title="23N Field Test #05 · Scheme questions in Gujarati") as demo:
        gr.Markdown(f"## Ask about a government scheme, in Gujarati or English\n{MODEL_LABEL} reads {N_DOCS} official documents. "
                    "Independent study on public data by 23N AI Labs.")
        q = gr.Textbox(label="Question", placeholder=EVAL[0][6])
        a = gr.Markdown(); s = gr.Markdown()
        gr.Button("Ask").click(gr_answer, q, [a, s]); q.submit(gr_answer, q, [a, s])
        gr.Examples([e[6] for e in EVAL[:4]] + [e[5] for e in EVAL[4:6]], q)
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", e)

## 8 · Video (1080×1080, 30 fps, 58 s, H.264) and cover (1080×1350)
Every number and answer in the video comes from the run above. The example question is the first one where the
English and Gujarati results differ (falling back to the first question), and the scorecard shows all 30 answers,
wrong ones included.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30
_GU_VIRAMA = "\u0ACD"

def grapheme_clusters(s):
    """Rough Gujarati-aware clusters so the typing animation never shows a dangling matra or half-conjunct."""
    out = []
    for ch in s:
        if out and (unicodedata.category(ch) in ("Mn", "Mc") or out[-1].endswith(_GU_VIRAMA) or ch in "\u200c\u200d"):
            out[-1] += ch
        else:
            out.append(ch)
    return out

def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def pick_example():
    by = {(r["qid"], r["lang"]): r for r in ROWS if r["mode"] == "rag"}
    for qid, *_ in EVAL:
        if by[(qid, "en")]["correct"] != by[(qid, "gu")]["correct"]:
            return by[(qid, "en")], by[(qid, "gu")]
    q0 = EVAL[0][0]
    return by[(q0, "en")], by[(q0, "gu")]

def _short(s, n=150):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + " …"

def _typed(text, t, a, b):
    cl = grapheme_clusters(text); k = int(round(len(cl) * _prog(t, a, b)))
    return "".join(cl[:k]) + ("|" if t < b else "")

def _badge(fig, r, y, alpha):
    ok = r["correct"]
    fig.text(0.055, y, "MATCHES THE SOURCE" if ok else "DOES NOT MATCH THE SOURCE",
             fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.MOSS if ok else B.AMBER, alpha=alpha, va="top")
    fig.text(0.055, y - 0.028, f"SOURCE SAYS: {r['reference']}".upper()[:90], fontproperties=B.FP.get("mono"),
             fontsize=13, color=B.MUTED, alpha=alpha, va="top")

def _answer_block(fig, label, r, t, t_type0, t_type1, t_ans, y):
    fig.text(0.055, y, label, fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.MUTED, va="top")
    q = r["question"]
    if has_gu(q):
        put_text_img(fig, _typed(q, t, t_type0, t_type1), 0.055, y - 0.03, 34, B.INK, bold=True, max_width=960)
    else:
        fig.text(0.055, y - 0.03, "\n".join(textwrap.wrap(_typed(q, t, t_type0, t_type1), 44)),
                 fontproperties=B.FP.get("display"), fontsize=30, color=B.INK, va="top", linespacing=1.1)
    if t >= t_ans:
        a = _prog(t, t_ans, t_ans + 0.6)
        put_text_img(fig, "Answer: " + _short(r["answer"]), 0.055, y - 0.155, 26, B.MOSS if r["correct"] else B.BODY,
                     max_width=960, alpha=a, max_lines=3)
        _badge(fig, r, y - 0.285, a)

def draw_evidence(fig, t, ctx):
    """t runs 0..35 inside the evidence section (video seconds 5..40)."""
    B.header(fig, FT_NUM)
    if t < 7:                                                         # the corpus
        fig.text(0.055, 0.85, "The documents", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, f"{N_DOCS} OFFICIAL DOCUMENTS · {len(PASSAGES)} PASSAGES · {N_SCHEMES} SCHEMES",
                 fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")
        ax = fig.add_axes([0.42, 0.17, 0.52, 0.58])
        vals = ctx["chunks"]; g = _ease(_prog(t, 0.3, 3.5))
        ax.barh(range(len(vals))[::-1], [v * g for v in vals], color=B.MOSS, height=0.62)
        ax.set_yticks(range(len(vals))[::-1], ctx["doc_labels"], fontsize=14)
        ax.set_xlim(0, max(vals) * 1.15 if max(vals) else 1); ax.tick_params(axis="x", labelsize=12)
        ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
        ax.set_xlabel("passages", fontsize=13); ax.grid(axis="y", visible=False)
        B.caption(fig, "Official scheme documents, mostly in English: PIB, PFRDA, Finance Ministry, Gujarat WCD")
    elif t < 22:                                                      # one question, two languages
        en, gu = ctx["ex_en"], ctx["ex_gu"]
        _answer_block(fig, "ASKED IN ENGLISH", en, t, 7.2, 9.0, 9.6, 0.85)
        if t >= 11:
            _answer_block(fig, "ASKED IN GUJARATI", gu, t, 11.2, 14.5, 15.3, 0.49)
        B.caption(fig, f"Same question, same documents. {ctx['model']} answers from the passages it retrieves")
    else:                                                             # scorecard + bars
        fig.text(0.055, 0.85, "All 30 answers", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        nshow = int(round(2 * N_Q * _prog(t, 22.3, 26)))
        for row, lang in enumerate(("en", "gu")):
            fig.text(0.055, 0.735 - row * 0.075, lang.upper(), fontproperties=B.FP.get("mono_bold"), fontsize=15,
                     color=B.MUTED, va="center")
            for i, ok in enumerate(ctx["marks"][lang]):
                if row * N_Q + i >= nshow: continue
                cx, cy = 0.12 + i * 0.055, 0.712 - row * 0.075
                fig.patches.append(matplotlib.patches.FancyBboxPatch((cx, cy), 0.045, 0.045, boxstyle="round,pad=0.003",
                                   transform=fig.transFigure, fc=(B.MOSS if lang == "en" else B.AMBER) if ok else B.PAPER_DEEP, ec="none"))
        if t > 26:
            ax = fig.add_axes([0.10, 0.24, 0.84, 0.34]); draw_results(ax, big=True)
            for p in ax.patches:
                p.set_height(p.get_height() * _ease(_prog(t, 26, 28.5)))
        B.caption(fig, "Filled = correct. Oracle = the right document handed over directly")

def render_video(ctx, path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "Can a small AI answer scheme questions in Gujarati?",
                f"{N_DOCS} official documents · {N_Q} questions × 2 languages · free GPU"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        key = ("c", min(round(t * 10) / 10, 3.6)) if t < 7 else (("q", round(t * 15) / 15) if t < 22 else ("s", min(round(t * 15) / 15, 28.6)))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t, ctx); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, f"{GU_CORRECT}/{N_Q}",
                               f"correct in Gujarati, against {EN_CORRECT}/{N_Q} in English, from the same documents",
                               f"{MODEL_LABEL} + {EMBED_LABEL} retrieval, scored against the source document"); v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, "Data: PIB, PFRDA, DFS, Gujarat WCD official documents"); v.hold(6)
    v.close()
    print(f"Rendered {path}: {v.frames} frames ({v.frames / FPS:.1f}s)")
    return path

def render_cover(ctx, path=COVER_PATH):
    B.cover(path, FT_NUM, "Can a small AI answer scheme questions in Gujarati?",
            f"{N_Q} questions · English vs Gujarati · official documents",
            f"{GU_CORRECT}/{N_Q}", f"correct in Gujarati, against {EN_CORRECT}/{N_Q} in English ({MODEL_LABEL})",
            image=lambda ax: (ax.set_axis_on(), draw_results(ax, big=True)),
            data_credit="Data: PIB, PFRDA, DFS, Gujarat WCD")
    return path

def build_video_ctx():
    ex_en, ex_gu = pick_example()
    by = {(r["qid"], r["lang"]): r["correct"] for r in ROWS if r["mode"] == "rag"}
    return {"ex_en": ex_en, "ex_gu": ex_gu, "model": MODEL_LABEL,
            "chunks": [int(chunks_per_doc[d]) for d in DOC_META],
            "doc_labels": [f"{d} · {DOC_META[d]['label']}" for d in DOC_META],
            "marks": {lang: [by[(e[0], lang)] for e in EVAL] for lang in ("en", "gu")}}

In [ ]:
# CELL: video_render
VCTX = build_video_ctx()
render_video(VCTX)
render_cover(VCTX)
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 9 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """Can a small open AI model answer a Gujarati question about a government scheme, using only the official documents?

23N Field Test #05: {headline}

Census 2011 counted about 5.5 crore people whose mother tongue is Gujarati. All {n_docs} official documents we used are in English.

How we tested it:
• {n_docs} documents from PIB, PFRDA, the Finance Ministry and Gujarat WCD, on {n_schemes} schemes
• {model_name} with {embed_name} retrieval, on a free Colab GPU
• The same {n_q} questions in English and Gujarati, checked against the source: English {en_correct}/{n_q}, Gujarati {gu_correct}/{n_q}
• Given the right document directly: English {en_oracle}/{n_q}, Gujarati {gu_oracle}/{n_q}

Example: "{example_q_gu}"
→ {example_answer_gu} {example_mark}

Limits: {n_q} questions per language, our translations, a number-matching scorer. A capability probe, not a benchmark, and not eligibility advice.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """Asked a {model_size} open AI model {n_q} questions about Indian government schemes, in English and in Gujarati, from official documents. English: {en_correct}/{n_q} right. Gujarati: {gu_correct}/{n_q}. 23N Field Test #05. Notebook below.

---
Thread (optional):
1/ Corpus: {n_docs} official documents (PIB, PFRDA, Finance Ministry, Gujarat WCD) on {n_schemes} schemes. Retrieval with {embed_name}, answers from {model_name} on a free GPU.
2/ Given the right document directly: English {en_oracle}/{n_q}, Gujarati {gu_oracle}/{n_q}. Right document retrieved: English {en_retrieval}/{n_q}, Gujarati {gu_retrieval}/{n_q}.
3/ Limits: {n_q} questions per language, our translations, number-matching scorer. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
ex_en, ex_gu = VCTX["ex_en"], VCTX["ex_gu"]
POST_VALUES = dict(headline=HEADLINE, n_q=N_Q, n_docs=N_DOCS, n_schemes=N_SCHEMES, model_name=MODEL_LABEL,
                   model_size=MODEL_SIZE, embed_name=EMBED_LABEL, en_correct=EN_CORRECT, gu_correct=GU_CORRECT,
                   en_oracle=EN_ORACLE, gu_oracle=GU_ORACLE, en_retrieval=EN_RET, gu_retrieval=GU_RET,
                   gu_script=GU_SCRIPT, median_latency=f"{LAT_MEDIAN:.1f}", hardware=HARDWARE,
                   example_q_gu=ex_gu["question"], example_answer_gu=_short(ex_gu["answer"], 120),
                   example_mark="✓" if ex_gu["correct"] else "✗", example_reference=ex_gu["reference"],
                   signature=B.post_signature(FT_NUM) + " #Gujarati #GovTech #OpenSourceAI")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 10 · Limitations (read before quoting the number)

- **Small, narrow test.** 15 questions per language over 8 documents on 7 schemes. Every question has a single
  numeric answer. Real citizen questions are vaguer ("am I eligible?"), mix languages and depend on personal
  circumstances. Read this as a capability probe, not a benchmark.
- **Our own questions and translations.** 23N AI Labs wrote the questions and translated them into Gujarati. A
  native speaker's phrasing, dialect or Gujlish would give different results.
- **Number-matching scorer.** An answer counts as correct if it contains the gold number(s), no stale number and
  few other numbers. It reads English number words and Gujarati ones from one to ten (બે લાખ → 200000), but it
  doesn't check the surrounding words, so a right number in a wrong sentence still scores.
- **English source documents.** The documents are in English, so the Gujarati condition tests cross-lingual
  retrieval plus answering in Gujarati. It doesn't test Gujarati-language source documents.
- **Documents change.** Scheme rules and premiums are revised. The sanity check stops the headline if a gold value
  is no longer in the downloaded text. Run it again before quoting it.
- **Not advice.** The model can be wrong. Nothing here is guidance on eligibility; the official portals are.
- **Licences.** The generator (`Qwen2.5-3B-Instruct`) is released under the non-commercial *Qwen Research* licence.